# Aivora AI - Smoke Test (300 steps)

Short, throwaway validation run before committing to the full run at
`configs/financial_poc.yaml`'s real scale. Answers with real measured
numbers, not estimates:

1. Does `batch_size=16` fit at `seq_len=1024` on whatever GPU is actually
   attached? Runs on a single GPU by default (proven - this is what
   produced `checkpoint_16000.pt`) and lets `torch.nn.DataParallel` engage
   automatically (`training/trainer.py`) only if more than one GPU happens
   to be present - this notebook does not require or gate on any specific
   GPU count.
2. What is the real interval-based tok/s throughput (the metric fixed to
   no longer be inflated by cumulative-tokens-over-this-run-only-elapsed-
   time)?

Uses `configs/financial_poc.yaml`'s real `batch_size`/`seq_len`/
`gradient_accumulation_steps`/`dataset_mix`/`dataset_token_overrides` -
none of that is edited. `max_steps`/`eval_interval` are overridden via
`train_model()`'s `max_steps_override`/`eval_interval_override`
parameters (not by editing the yaml). If `batch_size=16` OOMs, this
notebook tries exactly ONE fallback at half batch_size and reports
whichever real numbers it gets - it does not keep halving and guessing.

**This does NOT reproduce the full-scale data prep** - it prepares a
small, fast per-dataset token cap (still covering every bucket in the
real `dataset_mix`, same weights) since 300 steps of sampling-with-
replacement doesn't need the full ~126M-token corpus, and the two
questions above (memory fit, throughput) don't depend on which real
content is in the batch. Run the real `dataset_token_overrides`-scale
prep separately before the actual full run.

**Before running: Settings -> Internet -> On.** No specific accelerator
requirement - runs on whatever GPU is attached.


## 1. Environment

In [ ]:
import platform, sys, os
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())
print("Kaggle input mounted:", os.path.exists("/kaggle/input"), os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else [])


## 1b. GPU compute-capability check (before the first `import torch`)

Kaggle has been assigning this account a Tesla P100 (compute capability
6.0 / sm_60) instead of a T4, and the base image's shipped torch build
(2.10.0+cu128) only supports compute capability 7.0+ - every real CUDA
kernel launch fails with `AcceleratorError: no kernel image is available`
regardless of `batch_size`, which is what actually happened on the first
real attempt at this run.

Checked via `nvidia-smi` (not `torch.cuda.get_device_capability()`)
specifically so this runs **before** `import torch` - reinstalling torch
mid-process and `importlib.reload()`-ing it is not safe (see the main
training notebook's dependency-install cell for why: torch's C extension
re-registers native `TORCH_LIBRARY` namespaces with the dispatcher, which
crashes on a second registration). If the attached GPU needs an older,
wider-compatibility torch build, it's installed here, before torch is
ever imported for the first time - not reactively after a failure.

In [ ]:
import subprocess

nvidia_smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("nvidia-smi:", nvidia_smi.stdout.strip() or "(no output)", nvidia_smi.stderr.strip())

NEEDS_OLDER_TORCH = False
if nvidia_smi.returncode == 0 and nvidia_smi.stdout.strip():
    # First GPU's line, e.g. "Tesla P100-PCIE-16GB, 6.0"
    first_line = nvidia_smi.stdout.strip().splitlines()[0]
    name, _, cc_str = first_line.rpartition(",")
    try:
        compute_cap = float(cc_str.strip())
        if compute_cap < 7.0:
            NEEDS_OLDER_TORCH = True
            print(f"GPU '{name.strip()}' has compute capability {compute_cap} - below the "
                  "shipped torch build's minimum (7.0). Installing an older torch build "
                  "with wider compute-capability support before it's ever imported.")
        else:
            print(f"GPU '{name.strip()}' has compute capability {compute_cap} - "
                  "compatible with the shipped torch build, no reinstall needed.")
    except ValueError:
        print(f"Could not parse compute capability from {cc_str!r} - leaving the shipped "
              "torch build as-is and letting the GPU check cell catch any real problem.")
else:
    print("nvidia-smi query failed or returned nothing - leaving the shipped torch build "
          "as-is and letting the GPU check cell catch any real problem.")

if NEEDS_OLDER_TORCH:
    import sys
    # torch 2.7.1 (still within "2.7 or earlier", the last line that kept
    # Pascal/sm_60 kernels) + an older CUDA toolkit build (cu118) for the
    # widest realistic compute-capability coverage. This repo's own
    # attention implementation is hand-written (no scaled_dot_product_
    # attention / torch.compile dependency), so an older torch build is
    # not expected to break anything model-specific.
    reinstall = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "torch==2.7.1", "--index-url", "https://download.pytorch.org/whl/cu118"],
        capture_output=True, text=True,
    )
    print(reinstall.stdout[-3000:])
    print(reinstall.stderr[-3000:])
    if reinstall.returncode != 0:
        raise RuntimeError(
            "STATUS = BLOCKED: fallback torch==2.7.1+cu118 install failed for this "
            "compute-capability-6.0 GPU, see output above."
        )
    print("Installed torch==2.7.1+cu118 (not yet imported).")


## 2. GPU / CUDA verification (hard gate only on "no GPU at all")

Requires *a* GPU (any count, any type) - does not require or prefer a
specific GPU count. `torch.nn.DataParallel` in `training/trainer.py`
already only engages when `torch.cuda.device_count() > 1`, so a single GPU
just runs normally with no extra code path.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "STATUS = BLOCKED: torch.cuda.is_available() is False. "
        "Go to Settings (right sidebar) -> Accelerator and attach any GPU, "
        "save, and re-run this notebook from the top."
    )

GPU_COUNT = torch.cuda.device_count()
print("GPU count:", GPU_COUNT)
for i in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i}  {props.name}  {props.total_memory / 1024**3:.2f} GB VRAM  "
          f"compute {torch.cuda.get_device_capability(i)}")
print("torch version:", torch.__version__, "| CUDA build:", torch.version.cuda)

if GPU_COUNT > 1:
    print(f"{GPU_COUNT} GPUs available - DataParallel will engage automatically in train_model().")
else:
    print("1 GPU available - proceeding on single-GPU (the proven path; no DataParallel).")


## 3. Repository transfer + integrity check

In [ ]:
import subprocess, os

REPO_URL = "https://github.com/Ankushk-aosc/Aivora-AI.git"
REPO_DIR = "/kaggle/working/Aivora-AI"

if not os.path.exists(REPO_DIR):
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                             capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            "STATUS = BLOCKED: git clone failed (see stderr above). "
            "Most likely cause: Internet is off for this notebook "
            "(Settings -> Internet -> On), or the repo URL changed."
        )
else:
    print(f"{REPO_DIR} already present, skipping clone.")

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())

required_paths = [
    "models/model.py", "training/trainer.py", "training/data_loader.py",
    "data_sources/prepare.py", "evaluation/evaluator.py",
    "configs/financial_poc.yaml",
]
missing = [p for p in required_paths if not os.path.exists(p)]
if missing:
    raise RuntimeError(f"STATUS = BLOCKED: repo clone incomplete, missing {missing}")

# Confirm this clone actually has the DataParallel + override changes this
# smoke test depends on - fail loudly here rather than get a confusing
# TypeError deep inside train_model() if the push/pull didn't land.
import inspect
from training.trainer import train_model
sig = inspect.signature(train_model)
required_params = {"max_steps_override", "eval_interval_override"}
missing_params = required_params - set(sig.parameters)
if missing_params:
    raise RuntimeError(
        f"STATUS = BLOCKED: train_model() is missing {missing_params} - this "
        "clone does not have the smoke-test override support. Confirm the "
        "DataParallel/override commit was actually pushed to origin/main "
        "before re-running this notebook."
    )
print("Repo integrity check passed:", len(required_paths), "required paths present.")
print("train_model() signature:", sig)


## 4. Dependencies

In [ ]:
import subprocess, sys

with open("requirements.txt") as f:
    reqs = [line.strip() for line in f if line.strip() and not line.startswith("#")]

reqs_to_install = [r for r in reqs if not r.lower().startswith("torch")]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + reqs_to_install,
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
print(result.stderr[-3000:])
if result.returncode != 0:
    raise RuntimeError("STATUS = BLOCKED: pip install failed, see output above.")

if not torch.cuda.is_available():
    raise RuntimeError(
        "STATUS = BLOCKED: CUDA availability changed after installing "
        "requirements.txt - something in the dependency list broke it."
    )
print(f"Dependencies installed; CUDA still available after install ({torch.cuda.device_count()} GPU(s)).")


## 5. Small-scale smoke-test data prep

**NOT** `configs/financial_poc.yaml`'s real `dataset_token_overrides`
(that's 126.4M tokens total, meant for the real run) - this uses the same
`dataset_mix` buckets/weights but a small fixed per-dataset cap, since
300 steps of sampling-with-replacement only needs enough shards to fill
batches, not the full corpus. Run the real prep separately, before the
actual full run.

In [ ]:
import yaml
from data_sources.dataset_registry import list_entries
from data_sources.dataset_mixer import BUCKET_TO_CATEGORY, validate_mix
from data_sources.prepare import prepare_dataset

PRESET = "financial_poc"
SMOKE_TOKENS_PER_DATASET = 300_000  # small and fast - not the real budget

with open(f"configs/{PRESET}.yaml") as f:
    preset_cfg = yaml.safe_load(f)

mix = preset_cfg["dataset_mix"]
validate_mix(mix)
print(f"Preset '{PRESET}' dataset_mix (real weights, unedited): {mix}")
print(f"Smoke-test prep budget: {SMOKE_TOKENS_PER_DATASET:,} tokens per dataset (not the real dataset_token_overrides)")

summary = []
for bucket, weight in mix.items():
    category = BUCKET_TO_CATEGORY[bucket]
    entries = list_entries(category=category, verified_only=True)
    if not entries:
        print(f"  [SKIP] bucket '{bucket}' (category '{category}') has no VERIFIED datasets registered.")
        continue
    for entry in entries:
        shard_index = os.path.join("data", "shards", entry.name, "train", "index.json")
        if os.path.exists(shard_index):
            print(f"  '{entry.name}' already prepared, skipping.")
            continue
        print(f"  Preparing '{entry.name}' (bucket '{bucket}', budget {SMOKE_TOKENS_PER_DATASET:,} tokens)...")
        result = prepare_dataset(entry.name, max_tokens=SMOKE_TOKENS_PER_DATASET)
        summary.append((entry.name, result["train_tokens_used"], result["validation_tokens_used"]))

print()
print("Prepared datasets (real, measured token counts):")
for name, train_tok, val_tok in summary:
    print(f"  {name}: {train_tok:,} train / {val_tok:,} validation tokens")


## 6. Leakage check (hard gate)

In [ ]:
from evaluation import check_leakage

leak_report = check_leakage()
print(leak_report)
if not leak_report.get("clean", False):
    raise RuntimeError(f"STATUS = BLOCKED: leakage detected - {leak_report}")
print("Leakage check passed: no evaluation text found in training shards.")


## 7. Checkpoint discovery (checkpoint_16000.pt)

Resumes from `checkpoint_16000.pt`, uploaded as a Kaggle Dataset
(`aoscjkjhh/aivora-ai-checkpoint-16000`) rather than chained via
`kernel_sources` - a dataset mount has a predictable, guaranteed path
(`/kaggle/input/<dataset-slug>/<file>`), unlike `kernel_sources` output
mounting where the exact path prefix isn't knowable in advance (see the
main training notebook's own checkpoint-discovery cell for that
uncertainty). Add `aoscjkjhh/aivora-ai-checkpoint-16000` under
`dataset_sources` in this kernel's settings before running.

In [ ]:
import glob as _glob

RESUME_CHECKPOINT = "/kaggle/input/aivora-ai-checkpoint-16000/checkpoint_16000.pt"

if not os.path.exists(RESUME_CHECKPOINT):
    candidates = _glob.glob("/kaggle/input/**/checkpoint_16000.pt", recursive=True)
    if candidates:
        print(f"Guessed resume path not found; located checkpoint via search instead: {candidates[0]}")
        RESUME_CHECKPOINT = candidates[0]

from models import DeepSeekConfig, DeepSeekV3

if not os.path.exists(RESUME_CHECKPOINT):
    raise RuntimeError(
        f"STATUS = BLOCKED: {RESUME_CHECKPOINT} not found. Confirm the "
        "aoscjkjhh/aivora-ai-checkpoint-16000 dataset is attached under "
        "this kernel's Settings -> Add Data."
    )

ref_model = DeepSeekV3(DeepSeekConfig.default())
ref_state = ref_model.state_dict()

ckpt = torch.load(RESUME_CHECKPOINT, map_location="cpu")
ckpt_state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt

mismatches = []
for key, ref_tensor in ref_state.items():
    if key not in ckpt_state:
        mismatches.append(f"missing key: {key}")
    elif ckpt_state[key].shape != ref_tensor.shape:
        mismatches.append(f"shape mismatch on {key}: checkpoint has "
                           f"{ckpt_state[key].shape}, current model expects {ref_tensor.shape}")
if mismatches:
    raise RuntimeError(
        "STATUS = BLOCKED: checkpoint is not architecture-compatible with "
        f"the current model config: {mismatches[:5]}"
    )
print(f"Checkpoint {RESUME_CHECKPOINT} is architecture-compatible. Will resume from it.")
del ref_model


## 8. Smoke test: 300 steps at batch_size=4 (single clean attempt)

The fp16-vs-bf16 fix produced byte-identical peak memory to the original
bf16 run (14.08 GiB either way) - precision was not the cause of the OOM.
Real cause unconfirmed; trying a much smaller batch_size directly rather
than a small step down, to get a real signal in one attempt about whether
this model/config can fit on this P100 at all near financial_poc.yaml's
seq_len=1024, or whether something more structural is going on (e.g. the
MoE layer's per-expert Python loop in models/moe.py holding several
intermediate tensors live at once, inflating peak memory beyond what
101.7M parameters alone would suggest - not yet investigated).

Single clean attempt, same as the last two runs.

In [ ]:
import contextlib
import io
import re
import time
import warnings

from training.trainer import train_model

SMOKE_MAX_STEPS = 300
SMOKE_EVAL_INTERVAL = 100
TEST_BATCH_SIZE = 4  # precision fix made no difference (14.08 GiB either way) - try much smaller


def run_attempt(batch_size_override):
    n_gpus = torch.cuda.device_count()
    for i in range(n_gpus):
        torch.cuda.reset_peak_memory_stats(i)

    result = {"oom": False, "oom_error": None, "log_text": "", "warnings": [],
              "wall_elapsed": 0.0, "peak_mem": {}}
    captured_stdout = io.StringIO()
    start_wall = time.time()
    try:
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            with contextlib.redirect_stdout(captured_stdout):
                train_model(
                    preset_name="financial_poc",
                    resume=RESUME_CHECKPOINT,
                    max_steps_override=SMOKE_MAX_STEPS,
                    eval_interval_override=SMOKE_EVAL_INTERVAL,
                    batch_size_override=batch_size_override,
                )
            result["warnings"] = [str(warning.message) for warning in w]
    except torch.cuda.OutOfMemoryError as e:
        result["oom"] = True
        result["oom_error"] = str(e)
    result["wall_elapsed"] = time.time() - start_wall
    result["log_text"] = captured_stdout.getvalue()

    for i in range(n_gpus):
        result["peak_mem"][i] = {
            "allocated_gib": torch.cuda.max_memory_allocated(i) / 1024**3,
            "reserved_gib": torch.cuda.max_memory_reserved(i) / 1024**3,
            "total_gib": torch.cuda.get_device_properties(i).total_memory / 1024**3,
        }
    return result


print(f"Single clean attempt: batch_size={TEST_BATCH_SIZE}")
print("-" * 70)
attempt = run_attempt(batch_size_override=TEST_BATCH_SIZE)
print(attempt["log_text"])

tok_s_lines = re.findall(r"step (\d+): .*?\| (\d+) tok/s \|", attempt["log_text"])

print("=" * 70)
print("SMOKE TEST SUMMARY (single clean batch_size=8 attempt)")
print("=" * 70)
print("configs/financial_poc.yaml was NOT touched by this smoke test, regardless of outcome.")
if attempt["oom"]:
    print(f"RESULT: batch_size={TEST_BATCH_SIZE} OOM'd (clean, single-attempt measurement).")
    print(f"Error: {attempt['oom_error']}")
else:
    print(f"RESULT: batch_size={TEST_BATCH_SIZE} completed {SMOKE_MAX_STEPS} steps, no OOM.")
    print(f"Wall time: {attempt['wall_elapsed']:.1f}s")
    print(f"Interval tok/s per log line ({len(tok_s_lines)} line(s)):")
    for step, toks in tok_s_lines:
        print(f"  step {step}: {int(toks):,} tok/s")

print()
print("Peak GPU memory per device (torch.cuda.max_memory_allocated/reserved):")
for i, m in attempt["peak_mem"].items():
    print(f"  cuda:{i}  allocated={m['allocated_gib']:.2f} GiB  "
          f"reserved={m['reserved_gib']:.2f} GiB  (of {m['total_gib']:.2f} GiB total)")

if attempt["warnings"]:
    print(f"Warnings raised ({len(attempt['warnings'])}):")
    for msg in attempt["warnings"]:
        print(f"  - {msg}")
else:
    print("No warnings raised during this attempt.")
